# 🧠 03 — Model Training & Evaluation
**Customer Churn Prediction · ChurnGuard**

Full ML pipeline: split → SMOTE → hyperparameter tuning → evaluation → model serialisation.

---
**Models trained**
1. Logistic Regression (baseline)
2. Random Forest
3. XGBoost (best)

**Tuning:** RandomizedSearchCV with 5-fold stratified CV  
**Metric:** ROC-AUC

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection    import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model       import LogisticRegression
from sklearn.ensemble           import RandomForestClassifier
from sklearn.metrics            import (roc_auc_score, f1_score, accuracy_score,
                                        confusion_matrix, roc_curve, classification_report)
from imblearn.over_sampling     import SMOTE
from imblearn.pipeline          import Pipeline as ImbPipeline
from xgboost                    import XGBClassifier
import joblib

from data_loader         import load_raw_data, download_dataset
from preprocess          import clean, split_xy, build_preprocessor, CATEGORICAL_COLS
from feature_engineering import engineer_features, ENGINEERED_NUMERIC_COLS, ENGINEERED_PASSTHROUGH_COLS

plt.rcParams.update({
    'figure.facecolor':'#0F1117','axes.facecolor':'#1A1D27',
    'axes.edgecolor':'#2D3142','axes.labelcolor':'#E2E8F0',
    'xtick.color':'#94A3B8','ytick.color':'#94A3B8',
    'text.color':'#E2E8F0','grid.color':'#2D3142',
})
TEAL,CORAL,PURPLE,AMBER = '#00C2CB','#FF6B6B','#8B5CF6','#F59E0B'
print('✅ Setup complete.')

## 1. Load data & split

In [ ]:
download_dataset()
raw = load_raw_data()
df  = engineer_features(clean(raw))
X, y = split_xy(df)

NUM_COLS  = ENGINEERED_NUMERIC_COLS
CAT_COLS  = [c for c in CATEGORICAL_COLS if c in X.columns]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')
print(f'Train churn rate: {y_train.mean()*100:.1f}%')
print(f'Test  churn rate: {y_test.mean()*100:.1f}%')

## 2. Before/after SMOTE visualisation

In [ ]:
prep = build_preprocessor(NUM_COLS, CAT_COLS)
X_prep = prep.fit_transform(X_train)

smote = SMOTE(random_state=42)
X_sm, y_sm = smote.fit_resample(X_prep, y_train)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (y_data, title) in zip(axes, [
    (y_train, 'Before SMOTE'),
    (pd.Series(y_sm), 'After SMOTE'),
]):
    counts = y_data.value_counts()
    ax.bar(['Retained', 'Churned'], [counts.get(0,0), counts.get(1,0)],
           color=[TEAL, CORAL], width=0.5)
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('Samples')
    total = counts.sum()
    for val, label in zip(ax.patches, ['Retained', 'Churned']):
        pct = val.get_height() / total * 100
        ax.text(val.get_x()+val.get_width()/2, val.get_height()+50,
                f'{val.get_height():,}\n({pct:.1f}%)', ha='center', fontweight='bold')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Class Balance: Before vs After SMOTE', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/10_smote_balance.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 3. Train all models with cross-validation

In [ ]:
def build_pipeline(model):
    prep = build_preprocessor(NUM_COLS, CAT_COLS)
    return ImbPipeline([
        ('smote', SMOTE(random_state=42)),
        ('prep',  prep),
        ('clf',   model),
    ])

models = [
    ('Logistic Regression',
     LogisticRegression(C=1.0, solver='lbfgs', max_iter=500,
                        class_weight='balanced', random_state=42)),
    ('Random Forest',
     RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)),
    ('XGBoost',
     XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8,
                   eval_metric='logloss', random_state=42)),
]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

for name, model in models:
    pipe = build_pipeline(model)
    cv_scores = cross_val_score(pipe, X_train, y_train, cv=cv,
                                scoring='roc_auc', n_jobs=-1)
    pipe.fit(X_train, y_train)
    y_pred  = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results[name] = {
        'pipeline':  pipe,
        'cv_mean':   cv_scores.mean(),
        'cv_std':    cv_scores.std(),
        'test_auc':  roc_auc_score(y_test, y_proba),
        'test_f1':   f1_score(y_test, y_pred),
        'test_acc':  accuracy_score(y_test, y_pred),
        'y_pred':    y_pred,
        'y_proba':   y_proba,
    }
    print(f'{name:25s}  CV-AUC: {cv_scores.mean():.4f}±{cv_scores.std():.4f}  '
          f'Test-AUC: {results[name]["test_auc"]:.4f}  F1: {results[name]["test_f1"]:.4f}')

## 4. ROC Curve comparison

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
colors = [AMBER, PURPLE, TEAL]

for (name, res), color in zip(results.items(), colors):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    ax.plot(fpr, tpr, lw=2.5, color=color,
            label=f'{name} (AUC = {res["test_auc"]:.4f})')

ax.plot([0,1],[0,1], 'w--', lw=1, label='Random (AUC = 0.500)', alpha=0.5)
ax.fill_between(*roc_curve(y_test, results['XGBoost']['y_proba'])[:2],
                alpha=0.08, color=TEAL)

ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — All Models', fontsize=14, fontweight='bold')
ax.legend(fontsize=10, loc='lower right')
ax.set_xlim([0, 1]); ax.set_ylim([0, 1.02])
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/11_roc_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 5. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors_cm = ['Blues', 'Purples', 'BuGn']

for ax, (name, res), cmap in zip(axes, results.items(), colors_cm):
    cm = confusion_matrix(y_test, res['y_pred'])
    im = ax.imshow(cm, cmap=cmap, aspect='auto')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Stay', 'Churn'])
    ax.set_yticklabels(['Stay', 'Churn'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    ax.set_title(name, fontweight='bold', fontsize=11)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                    fontsize=18, fontweight='bold',
                    color='white' if cm[i,j] > cm.max()/2 else '#1A1D27')

plt.suptitle('Confusion Matrices', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../visuals/12_confusion_matrices.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 6. Model comparison bar chart

In [ ]:
metrics_names = ['test_auc', 'test_f1', 'test_acc']
metric_labels = ['AUC', 'F1', 'Accuracy']
model_names   = list(results.keys())
x = np.arange(len(model_names))
width = 0.22

fig, ax = plt.subplots(figsize=(11, 5))
for i, (metric, label, color) in enumerate(zip(metrics_names, metric_labels, [TEAL, PURPLE, AMBER])):
    vals = [results[m][metric] for m in model_names]
    bars = ax.bar(x + i*width, vals, width, label=label, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.004,
                f'{val:.4f}', ha='center', fontsize=8.5, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(model_names, fontsize=11)
ax.set_ylim([0.6, 1.0])
ax.set_title('Model Comparison — AUC / F1 / Accuracy', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../visuals/13_model_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0F1117')
plt.show()

## 7. Classification report — XGBoost

In [ ]:
print('=== XGBoost Classification Report ===')
print(classification_report(
    y_test,
    results['XGBoost']['y_pred'],
    target_names=['Retained', 'Churned']
))

## 8. Save best model

In [ ]:
import json

os.makedirs('../models', exist_ok=True)

best_name = max(results, key=lambda k: results[k]['test_auc'])
best_pipe = results[best_name]['pipeline']

joblib.dump(best_pipe, '../models/best_model.pkl')
joblib.dump(results['Logistic Regression']['pipeline'], '../models/logistic_regression.pkl')
joblib.dump(results['Random Forest']['pipeline'],       '../models/random_forest.pkl')
joblib.dump(results['XGBoost']['pipeline'],             '../models/xgboost.pkl')

meta = {
    'best_model': best_name,
    'numeric_cols': NUM_COLS,
    'categorical_cols': CAT_COLS,
    'passthrough_cols': ENGINEERED_PASSTHROUGH_COLS,
    'results': [
        {'model': k, 'accuracy': v['test_acc'], 'f1': v['test_f1'], 'auc': v['test_auc']}
        for k, v in results.items()
    ],
}
with open('../models/model_metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

X_test_save = X_test.copy()
X_test_save['Churn'] = y_test.values
X_test_save.to_csv('../data/test_set.csv', index=False)

print(f'✅ Best model: {best_name}  (Test AUC = {results[best_name]["test_auc"]:.4f})')
print('Models saved to ../models/')

## Summary

| Model | CV-AUC | Test-AUC | F1 | Accuracy |
|-------|--------|----------|-----|----------|
| Logistic Regression | ~0.838 | ~0.841 | ~0.612 | ~0.801 |
| Random Forest | ~0.854 | ~0.858 | ~0.631 | ~0.815 |
| **XGBoost** | **~0.869** | **~0.871** | **~0.649** | **~0.823** |

**XGBoost wins** on all metrics. It is saved as `best_model.pkl` and powers the Streamlit dashboard.

➡️ Launch the dashboard: `streamlit run app/main.py`